# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [ ]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/hub/"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")

## 2. Loading Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
# model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "openai-community/gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
# TODO: Check why dtype = auto solved the problem
# TODO: what is the default value of torch_dtype -> look in the githubb documentation
# Always use "auto"
model.NAME = model_name

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

In [ ]:
# Example inference

from transformers import AutoTokenizer
import transformers 
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

prompt = "What famous tower is in Paris?"
formatted_prompt = (
    f"### Human: {prompt}### Assistant:"
)

sequences = pipeline(
    formatted_prompt,
    do_sample=True,
    top_k=50,
    top_p = 0.7,
    num_return_sequences=1,
    repetition_penalty=1.1,
    max_new_tokens=500,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


## 3. Loading Datasets

### 3.1. WikiText

In [ ]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

# wikitext_sequence_length = tokenizer.model_max_length
# wikitext_sequence_length = 10
wikitext_sequence_length = 2048
wikitext_batch_size = 1  # Just use batch size 1 for this project
wikitext_stride = 2048
wikitext_seed = 3
# wikitext_n_lines = 406
wikitext_n_lines = None

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=wikitext_batch_size,
  sequence_length=wikitext_sequence_length,
  stride=wikitext_stride,
  tokenizer_name=model_name,
  seed=wikitext_seed,
  n_lines = wikitext_n_lines
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

### 3.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

directory_dataset = os.getcwd()
oasst_batch_size = 1  # Just use batch size 1 for this project
# oasst_batch_size = 16
# oasst_batch_size = 64
oasst_sequence_length = 512  # Maximum sequence length - use the default value
oasst_seed = 1

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=directory_dataset,
  batch_size=oasst_batch_size,
  sequence_length=oasst_sequence_length,
  tokenizer_name=model_name,
  seed=oasst_seed
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 4. Quantization

### 4.3 HQQ

In [ ]:
# HQQ Config
from transformers import AutoTokenizer

# Define the model name and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### 4.3.1 Option 1: All linear layers will use the same quantization config

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig
from src import MODEL_SAVE_PATH

# Option 1: All linear layers will use the same quantization config
quant_config_1 = HqqConfig(
    nbits=8, 
    group_size=64, 
    quant_zero=False, 
    quant_scale=False, 
    axis=0  # Default value
)

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
# Quantize the model with the same config for all linear layers
hqq_model_1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quant_config_1
)

# Save path
import os
hqq_model_name = f"{model_name.split('/')[1]}-hqq-1"
hqq_model_path = os.path.join(MODEL_SAVE_PATH, hqq_model_name)
hqq_model_1.NAME = hqq_model_name
hqq_model_1.PATH = hqq_model_path
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Saving the model is not possible due to the current version of transformers
# ValueError: The model is quantized with hqq and is not serializable -
# check out the warnings from the logger on the traceback to understand the reason why the quantized model is not serializable.
# ValueError: .to is not supported for HQQ-quantized models.

### 4.3.2 Option 2: Different configs for specific layers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, HqqConfig
from src import MODEL_SAVE_PATH

# Option 2: Different configs for specific layers
q4_config = {'nbits': 4, 'group_size': 64, 'quant_zero': False, 'quant_scale': False}
q3_config = {'nbits': 3, 'group_size': 32, 'quant_zero': False, 'quant_scale': False}

quant_config_2 = HqqConfig(dynamic_config={
    'self_attn.q_proj': q4_config,
    'self_attn.k_proj': q4_config,
    'self_attn.v_proj': q4_config,
    'self_attn.o_proj': q4_config,
    'mlp.gate_proj': q3_config,
    'mlp.up_proj': q3_config,
    'mlp.down_proj': q3_config,
})

# Quantize the model with different configs for specific layers
hqq_model_2 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    quantization_config=quant_config_2
)

# Save path
import os
hqq_model_name = f"{model_name.split('/')[1]}-hqq-2"
hqq_model_path = os.path.join(MODEL_SAVE_PATH, hqq_model_name)
hqq_model_2.NAME = hqq_model_name
hqq_model_2.PATH = hqq_model_path
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

### 4.3.3 Option 3: Using the HQQ library

In [ ]:
from transformers import AutoModelForCausalLM
from hqq.models.hf.base import AutoHQQHFModel
from hqq.core.quantize import BaseQuantizeConfig as HQQBaseQuantizeConfig
import os
from src import MODEL_SAVE_PATH
hqq_model_3 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device="cuda")

#Quantize
quant_config_3 = HQQBaseQuantizeConfig(nbits=4, group_size=64, quant_scale=False, quant_zero=False, axis=1) 
AutoHQQHFModel.quantize_model(hqq_model_3, quant_config=quant_config_3, compute_dtype=torch.float16, device=device)

# Save model
hqq_model_name = f"{model_name.split('/')[1]}-hqq-3"
hqq_model_path = os.path.join(MODEL_SAVE_PATH, hqq_model_name)
hqq_model_3.NAME = hqq_model_name
hqq_model_3.PATH = hqq_model_path
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

#Save: Make sure to save the model BEFORE any patching
AutoHQQHFModel.save_quantized(hqq_model_3, hqq_model_path)
#Load
# model = AutoHQQHFModel.from_quantized(save_dir)

### 4.3.4 Option 4: Using HQQ Library with LoRa Fine-Tuning

In [ ]:
#First, quantize/load a quantized HQQ model the
from hqq.core.peft import PeftUtils

base_lora_params = {
  'lora_type':'default',
  'r': 32,
  'lora_alpha': 64,
  'dropout': 0.05,
  'train_dtype': torch.float32
}

lora_params = {
  'self_attn.q_proj': base_lora_params,
  'self_attn.k_proj': base_lora_params,
  'self_attn.v_proj': base_lora_params,
  'self_attn.o_proj': base_lora_params,
  'mlp.gate_proj': None,
  'mlp.up_proj': None,
  'mlp.down_proj': None
}

#Add LoRA to linear/HQQ modules
print("Adding LoRA to model")
PeftUtils.add_lora(model, lora_params)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

#Train ....

#Convert LoRA weights to the same model dtype for faster inference
model.eval()
print("Model in eval mode")
print("Casting LoRA weights to model dtype")
PeftUtils.cast_lora_weights(model, dtype=torch.float32)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
hqq_lora_save_path = os.path.join(MODEL_SAVE_PATH, "hqq_lora")
#Save LoRA weights
PeftUtils.save_lora_weights(model, hqq_lora_save_path)

#Load LoRA weights: automatically calls add_lora 
PeftUtils.load_lora_weights(model, hqq_lora_save_path)

In [ ]:
print(model_name)
print(tokenizer.pad_token)
print(tokenizer.eos_token)
print(tokenizer.padding_side)
print(tokenizer.add_bos_token)
print(tokenizer.add_eos_token)

In [ ]:
# Settings
######################################################################################
hf_auth = None  # HuggingFace token
cache_path = None  # cache directory to store data

# HQQ Quantize
######################################################################################
from hqq.engine.hf import HQQModelForCausalLM, AutoTokenizer

print("Loading model")
model = HQQModelForCausalLM.from_pretrained(model_name, cache_dir=cache_path)
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True, padding=True, cache_dir=cache_path)

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Quantize the model
from hqq.core.quantize import *

hqqplus_params = {
    'nbits': 4,
    'group_size': 64,
    'quant_scale': False,
    'quant_zero': False
}

print("Quantizing model")
quant_config = BaseQuantizeConfig(**hqqplus_params)
model.quantize_model(quant_config=quant_config)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Add Peft
######################################################################################
from hqq.core.peft import PeftUtils
from hqq.core.quantize import *

import torch

train_dtype = torch.float32
base_lora_params = {
    'lora_type': 'default',
    'r': 32,
    'lora_alpha': 64,
    'dropout': 0.05,
    'train_dtype': train_dtype
}

lora_params = {
    'self_attn.q_proj': base_lora_params,
    'self_attn.k_proj': base_lora_params,
    'self_attn.v_proj': base_lora_params,
    'self_attn.o_proj': base_lora_params,
    'mlp.gate_proj': None,
    'mlp.up_proj': None,
    'mlp.down_proj': None
}

# Apply LoRA
print("Adding LoRA to model")
PeftUtils.add_lora(model, lora_params)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Dataset
######################################################################################
from datasets import load_dataset, Dataset
from tqdm import tqdm
import transformers
import numpy as np
import random

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.add_bos_token = False
tokenizer.add_eos_token = False

batch_size = 1
num_epochs = 1
grad_acc = 1
max_tokens = 256
max_samples = 5000

print("Loading dataset")
# dataset_train = load_dataset("OpenAssistant/oasst1", split="train")
# dataset_val = load_dataset("OpenAssistant/oasst1", split="validation")
dataset_train = load_dataset("timdettmers/openassistant-guanaco", split="train")
dataset_val = load_dataset("timdettmers/openassistant-guanaco", split="test")

def pre_process_chat(chat):
    return chat

random.seed(100)
idx = random.sample(range(len(dataset_train)), min(max_samples, len(dataset_train)))
dataset_train = Dataset.from_dict({'text': [pre_process_chat(dataset_train[i]['text']) for i in tqdm(idx)]})
dataset_val = Dataset.from_dict({'text': [pre_process_chat(dataset_val[i]['text']) for i in range(len(dataset_val))]})

#####################################################################################
# Train
from trl import SFTTrainer

grad_acc = 2
logging_st = 1
max_steps = -1
lr = 1e-4
batch_size = 1
n_epochs = 1

print("Setting up training")
training_args = transformers.TrainingArguments(
    output_dir='.',
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_acc,
    learning_rate=lr,
    logging_steps=logging_st,
    num_train_epochs=n_epochs,
    max_steps=max_steps,
    remove_unused_columns=False,
    fp16=train_dtype == torch.float32,
    max_grad_norm=1.0,
    save_steps=10000000,
    lr_scheduler_type="linear",
)

class WrappedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, *args, **kwargs):
        return self.model.forward(*args, **kwargs)

    def train(self):
        self.model.train()

    def eval(self):
        self.model.eval()

    def parameters(self):
        return self.model.parameters()

trainer = SFTTrainer(
    model=WrappedModel(model),
    tokenizer=tokenizer,
    max_seq_length=max_tokens,
    train_dataset=dataset_train,
    eval_dataset=None,
    peft_config=None,
    packing=True,
    args=training_args,
    dataset_text_field="text",
)

model.is_parallelizable = False
trainer.is_model_parallel = False
trainer.place_model_on_device = False

print("Training model")
model.train()
trainer.train()
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
dataset_train['text']

In [ ]:
from src import MODEL_SAVE_PATH

#Convert LoRA weights to the same model dtype for faster inference
model.eval()
print("Model in eval mode")
print("Casting LoRA weights to model dtype")
PeftUtils.cast_lora_weights(model, dtype=torch.float32)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
hqq_lora_save_path = os.path.join(MODEL_SAVE_PATH, "TinyLlama-hqq-lora")
print(f"Saving model to {hqq_lora_save_path}")
#Save LoRA weights
PeftUtils.save_lora_weights(model, hqq_lora_save_path)

#Load LoRA weights: automatically calls add_lora 
PeftUtils.load_lora_weights(model, hqq_lora_save_path)

In [ ]:
from src import MODEL_SAVE_PATH

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

hqq_lora_save_path = os.path.join(MODEL_SAVE_PATH, "TinyLlama-hqq-lora")
PeftUtils.load_lora_weights(model, hqq_lora_save_path)

In [ ]:
from datasets import load_dataset, Dataset

# dataset = load_dataset("timdettmers/openassistant-guanaco", split="train") 
# dataset_2 = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
# dataset_3 = load_dataset("OpenAssistant/oasst1", split="train")
print(dataset_train['text'])
# print(dataset['text'][0])
# print(dataset_2['text'][1])

In [ ]:
for string in dataset_3['text']:
    print(len(string))
print(len(dataset_3['text']))

## 5. Evaluation

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

### 5.1. Perplexity

In [ ]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# Evaluate Perplexity
print("\n################################")
print("Evaluating Perplexity...")
print("################################\n")

def evaluate_perplexity(model, dataloader, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Metric on current batch
            perplexity = metric(logits.float(), y)
            print(f"Perplexity: {perplexity:.2f}")

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"\nFinal Perplexity (PPL): {perplexity:.3f}")
    return perplexity.item()

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 5.2. Brier Score

In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)